# 🧩 Полный анализ данных: отчёты, граф зависимостей

Этот ноутбук выполняет:
1. Загрузку CSV‑файлов (кодировка cp1251, разделитель `;`).
2. Очистку и обогащение данных (создание `Full Name`, `Ingoing References` и т.д.).
3. Генерацию текстовых отчётов (через кнопки скачивания).

## Загрузка данных

1. Перетащите ваши CSV-файлы (`Cubes.csv`, `Multicubes.csv`, `Lists.csv`, `ListsProperties.csv`, `CubesSubsets.csv`, `ListsSubsets.csv`) в боковую панель JupyterLite (левая часть экрана, область "Files").
2. После того как файлы появятся в списке, выполните следующую ячейку — она прочитает их с правильной кодировкой cp1251.

#### Критическое значение количества клеток для куба

В следующей ячейке можно задать значение минимального количества клеток для кубов, которые попадут в специализированные отчеты.

Например, если вы хотите получить отчет о кубах, где количество ячеек больше 100 тысяч, задайте следующее значение:

```py
cube_threshold_value = 100_000
```

Значение по умолчанию - 500 тысяч ячеек.

In [ ]:
cube_threshold_value = 500_000

In [ ]:
# === 0. Установка дополнительных пакетов (выполняется один раз в сессии) ===
import piplite
await piplite.install('networkx')
await piplite.install('matplotlib')
await piplite.install('ipywidgets')
print("✅ Пакеты networkx, matplotlib, ipywidgets установлены")

In [ ]:
# === 1. Импорт библиотек
import pandas as pd
import numpy as np
import re
import io
import sys
import csv
import base64
import json
from collections import defaultdict
import networkx as nx
import matplotlib.pyplot as plt
import ipywidgets as widgets
from decimal import Decimal
from IPython.display import display, HTML

In [ ]:
# === 2. Функция для скачивания отчета
def download_text(content, filename):
    b64 = base64.b64encode(content.encode('utf-8')).decode()
    href = f'<a download="{filename}" href="data:text/plain;charset=utf-8;base64,{b64}">📥 Скачать {filename}</a>'
    display(HTML(href))

In [ ]:
# === 2. Функция для скачивания отчета
def download_csv(content, filename):
    """
    Скачивание CSV-файла с поддержкой кириллицы.
    
    Параметры:
    content (str): строка с содержимым CSV (без BOM)
    filename (str): имя файла, должно оканчиваться на .csv
    """
    # Добавляем BOM (Byte Order Mark) для корректного открытия в Excel с кириллицей
    content_with_bom = '\ufeff' + content
    # Кодируем в base64
    b64 = base64.b64encode(content_with_bom.encode('utf-8')).decode()
    # Формируем data-URL с правильным MIME-типом text/csv
    href = f'<a download="{filename}" href="data:text/csv;charset=utf-8;base64,{b64}">📥 Скачать {filename}</a>'
    display(HTML(href))

In [ ]:
# === 3. Функция разбора имен и функция проверки списка

def parse_name(s: str):
    s = str(s)
    parts = []
    i = 0
    n = len(s)

    while i < n and len(parts) < 2:
        if s[i] == "'":
            # Кавыченная часть: ищем закрывающую кавычку, после которой конец или точка
            j = i + 1
            while j < n:
                if s[j] == "'" and (j + 1 == n or s[j + 1] == '.'):
                    parts.append(s[i:j+1])
                    i = j + 1
                    if i < n and s[i] == '.':
                        i += 1
                    break
                j += 1
            else:
                # Кавычка не закрыта — остаток строки
                parts.append(s[i:])
                break
        else:
            # Некавыченная часть до точки
            j = i
            while j < n and s[j] != '.':
                j += 1
            parts.append(s[i:j])
            i = j
            if i < n and s[i] == '.':
                i += 1

    first = parts[0] if parts else ''
    second = parts[1] if len(parts) > 1 else None
    return first, second

def has_items(value):
    """Возвращает True, если значение содержит хотя бы один элемент."""
    if value is None:
        return False
    # Числовой NaN
    if isinstance(value, float) and np.isnan(value):
        return False
    # Массив NumPy (в т.ч. пустой)
    if isinstance(value, np.ndarray):
        return value.size > 0
    # Список или кортеж
    if isinstance(value, (list, tuple)):
        return len(value) > 0
    # Строка (пустая или '[]')
    if isinstance(value, str):
        return value.strip() not in ('', '[]', '{}')
    # Остальные типы (числа, bool и т.д.) – преобразуем
    try:
        return bool(value)
    except ValueError:
        return False

In [ ]:
# === 4. Чтение всех исходных данных

def read_cp1251_csv(filename, sep=';', has_id=True):
    with open(filename, 'rb') as f:
        raw = f.read()
    text = raw.decode('cp1251')
    if text.startswith('\ufeff'):
        text = text[1:]

    # Читаем CSV: для файлов с Id используем converters, чтобы прочитать Id как строку
    converters = {'Id': str} if has_id else None
    df = pd.read_csv(io.StringIO(text), sep=sep, converters=converters)

    # Если ожидается Id и колонка есть — восстанавливаем полное целое число
    if has_id and 'Id' in df.columns:
        def clean_id(x):
            if pd.isna(x):
                return ''
            s = str(x).strip()
            try:
                # Преобразуем экспоненциальную запись через Decimal (точное восстановление)
                d = Decimal(s)
                # Если число целое (например, 102000000000), возвращаем строку без .0
                if d == d.to_integral_value():
                    return str(int(d))
                else:
                    return s
            except:
                return s
        df['Id'] = df['Id'].apply(clean_id)
    elif has_id and 'Id' not in df.columns:
        print(f"⚠️ В файле {filename} нет колонки 'Id', но ожидалась.")
    
    return df

# Список файлов: (имя, есть ли Id?)
file_spec = [
    ('Cubes.csv', True),
    ('Multicubes.csv', True),
    ('Lists.csv', True),
    ('ListsProperties.csv', True),
    ('CubesSubsets.csv', False),
    ('ListsSubsets.csv', False)
]

dataframes = {}

for fname, has_id in file_spec:
    try:
        df = read_cp1251_csv(fname, has_id=has_id)
        dataframes[fname] = df
        id_status = "с Id" if has_id else "без Id"
        print(f"✅ Загружен {fname} — строк: {len(df)} ({id_status})")
    except FileNotFoundError:
        print(f"⚠️ Файл {fname} не найден")
    except Exception as e:
        print(f"❌ Ошибка при чтении {fname}: {e}")

# Проверка, что все нужные для поиска файлы загружены
required_for_search = ['Cubes.csv', 'Multicubes.csv', 'Lists.csv', 'ListsProperties.csv']
missing = [f for f in required_for_search if f not in dataframes]
if missing:
    print(f"\n❗ Отсутствуют файлы, необходимые для поиска: {missing}")
else:
    print("\n✅ Все файлы с Id успешно загружены. Id восстановлены из экспоненциальной формы.")

## 📊 Часть 1: Очистка и обогащение данных

In [ ]:
def rename_first_column(df, new_name):
    """Переименовывает первый столбец DataFrame в new_name"""
    if len(df.columns) > 0:
        cols = list(df.columns)
        cols[0] = new_name
        df.columns = cols
    return df

# === Функции очистки (оригинал из model_data) ===
def clean_df_cubes(df):
    """Очистка Cubes: удаление пустых Multicube, создание Full Name"""
    df = df.dropna(subset=['Multicube'])
    df = df[df['Multicube'].str.strip() != '']
    df['Cubes'] = df['Cubes'].fillna('').str.strip()
    df['Multicube'] = df['Multicube'].str.strip()
    df['Full Name'] = df['Multicube'] + '.' + df['Cubes']
    df = df[df['Full Name'] != '.']
    
    def quote_if_needed(val):
        if pd.isna(val) or val == '':
            return "''"
        if isinstance(val, str) and re.fullmatch(r'[a-zA-Z0-9_]+', val):
            return val
        else:
            return f"'{val}'"
    
    processed_multicube = df['Multicube'].apply(quote_if_needed)
    processed_cubes = df['Cubes'].apply(quote_if_needed)
    df['Full Name'] = processed_multicube + '.' + processed_cubes
    cols = list(df.columns)
    multicube_idx = cols.index('Multicube')
    cols.insert(multicube_idx + 1, cols.pop(cols.index('Full Name')))
    df = df[cols]
    return df

def clean_df_multicubes(df):
    """Очистка Multicubes: удаление строк с пустыми User Lists, Time Scale и Cell Count == 0"""
    required = ['User Lists', 'Time Scale', 'Cell Count', 'Multicubes']
    for col in required:
        if col not in df.columns:
            df[col] = ''
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce').fillna(0)
    mask = (df['User Lists'].isna() | (df['User Lists'] == '')) & \
           (df['Time Scale'].isna() | (df['Time Scale'] == '')) & \
           (df['Cell Count'] == 0)
    df = df[~mask].reset_index(drop=True)
    
    def process_name(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['Multicubes'].apply(process_name)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_df_lists(df):
    """Очистка Lists: удаление строк с Cell Count == 0 и Element Count == 0"""
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce').fillna(0)
    df['Element Count'] = pd.to_numeric(df['Element Count'], errors='coerce').fillna(0)
    mask = (df['Cell Count'] == 0) & (df['Element Count'] == 0)
    df = df[~mask].reset_index(drop=True)
    def process_name(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['Lists'].apply(process_name)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_df_listsproperties(df):
    """
    Очистка ListsProperties: создание List и Full Name на основе Format и ListsProperties.
    
    Логика (строго по вашему примеру):
    - Строки, где Format = NaN/пустое — это "уровни" (контейнеры), они **не попадают в результат**.
    - Строки, где Format ≠ NaN/пустое — это **элементы**, они **попадают в результат**.
    - Для каждого элемента:
        * List = последний **непустой** ListsProperties **до текущей строки** (в порядке следования)
        * Value = текущий ListsProperties (даже если он == Format)
        * Full Name = 'List.Value'
    - Пустое значение: NaN, None, '', 'null', 'NULL', 'n/a', 'N/A' (регистронезависимо)
    """
    # Копируем датафрейм, чтобы не менять оригинал
    df = df.copy()

    # Убедимся, что нужные столбцы существуют
    required = ['Format', 'ListsProperties']
    for col in required:
        if col not in df.columns:
            df[col] = ''

    # Добавим временные столбцы
    df['List'] = ''
    df['Full Name'] = ''

    # Функция для проверки, является ли значение "пустым"
    def is_empty(val):
        if pd.isna(val):
            return True
        if not isinstance(val, str):
            return False
        stripped = val.strip()
        return stripped == '' or stripped.lower() in ('null', 'n/a', 'na')

    # Функция для безопасного квотирования
    def quote_if_needed(val):
        if is_empty(val):
            return "''"
        if isinstance(val, str):
            stripped = val.strip()
            if re.fullmatch(r'[a-zA-Z0-9_]+', stripped):
                return stripped
            else:
                return f"'{stripped}'"
        return str(val)

    # ✅ Проходим по ВСЕМ строкам (включая пустые Format)
    last_valid_list = None  # Последний непустой ListsProperties (любой, даже если он == Format)

    for idx in range(len(df)):
        curr_format = df.loc[idx, 'Format']
        curr_listsprop = df.loc[idx, 'ListsProperties']

        # ✅ Обновляем last_valid_list, если ListsProperties НЕ пустое (если Format ПУСТ)
        if not is_empty(curr_listsprop) and is_empty(curr_format):
            last_valid_list = curr_listsprop

        # ✅ Если Format НЕ пуст — это элемент, заполняем Full Name
        if not is_empty(curr_format):
            value = curr_listsprop
            list_val = last_valid_list if last_valid_list is not None else ''

            df.loc[idx, 'List'] = list_val
            df.loc[idx, 'Full Name'] = f"{quote_if_needed(list_val)}.{quote_if_needed(value)}"

    # ✅ Фильтруем: оставляем ТОЛЬКО строки с НЕПУСТЫМ Format
    df = df[~df['Format'].apply(is_empty)].reset_index(drop=True)

    # ✅ Перемещаем столбцы 'List' и 'Full Name' сразу после первого столбца
    cols = list(df.columns)
    if len(cols) > 0:
        first_col = cols[0]
        for col in ['List', 'Full Name']:
            if col in cols:
                cols.remove(col)
        first_idx = cols.index(first_col) + 1
        cols.insert(first_idx, 'List')
        cols.insert(first_idx + 1, 'Full Name')
        df = df[cols]

    return df

def clean_df_сubesubsets(df):
    """Очистка CubesSubsets: создание Full Name"""
    def process(val):
        if pd.isna(val) or val == '':
            return ''
        if re.fullmatch(r'[\w]+', str(val)):
            return str(val)
        else:
            return f"'{val}'"
    df['Full Name'] = df['CubesSubsets'].apply(process)
    cols = list(df.columns)
    if 'Full Name' in cols:
        cols.remove('Full Name')
        cols.insert(1, 'Full Name')
        df = df[cols]
    return df

def clean_referenced_by(df, context_col='List'):
    """Обработка столбца Referenced By → Outgoing References"""
    if 'Referenced By' not in df.columns:
        df['_Out Refs'] = [[] for _ in range(len(df))]
        df['Outgoing References'] = [[] for _ in range(len(df))]
        return df
    def quote_if_needed(val):
        if pd.isna(val) or val == '':
            return "''"
        if isinstance(val, str) and re.fullmatch(r'[a-zA-Z0-9_]+', val):
            return val
        else:
            return f"'{val}'"
    outgoing = []
    for idx, row in df.iterrows():
        ref_str = row['Referenced By']
        context = row[context_col] if context_col in df.columns else ''
        if pd.isna(ref_str) or ref_str == '':
            outgoing.append([])
            continue
        items = re.findall(r"'([^']*)'\.'([^']*)'|([a-zA-Z0-9_]+)|'([^']*)'", ref_str)
        refs = []
        for match in items:
            if match[0] and match[1]:
                refs.append(f"{match[0]}.{match[1]}")
            elif match[2]:
                refs.append(match[2])
            elif match[3]:
                refs.append(match[3])
        # нормализация с контекстом
        norm = []
        for r in refs:
            if '.' in r:
                norm.append(r)
            else:
                norm.append(f"{quote_if_needed(context)}.{r}")
        outgoing.append(norm)
    df['Outgoing References'] = outgoing
    df['_Out Refs'] = [[x.replace("'", "") for x in inner] for inner in outgoing]
    return df

def add_ingoing_references(dataframes_dict):
    """
    Добавляет столбец 'Ingoing References' для всех DataFrame в словаре.
    
    Параметры:
        dataframes_dict (dict): словарь вида {имя_файла: pd.DataFrame}
    
    Возвращает:
        dict: тот же словарь с модифицированными датафреймами (изменение in-place)
    """
    # Сбор всех пар (источник → цель) из всех датафреймов
    outgoing_pairs = []
    for df in dataframes_dict.values():
        if 'Full Name' in df.columns and 'Outgoing References' in df.columns:
            for _, row in df.iterrows():
                src = row['Full Name']
                # Обрабатываем 'Outgoing References'
                for tgt in row['Outgoing References']:
                    outgoing_pairs.append((src, tgt))
                # Обрабатываем '_Out Refs' (очищенная версия, если есть)
                for tgt in row.get('_Out Refs', []):
                    outgoing_pairs.append((src, tgt))
    
    # Построение обратного индекса: цель → множество источников
    ingoing_index = defaultdict(set)
    for src, tgt in outgoing_pairs:
        ingoing_index[tgt].add(src)
    
    # Добавление столбца 'Ingoing References' в каждый датафрейм
    for df in dataframes_dict.values():
        if 'Full Name' not in df.columns:
            continue
        ingoing_list = []
        for _, row in df.iterrows():
            cur = row['Full Name']
            sources = ingoing_index.get(cur, set()).copy()
            sources.discard(cur)  # удаляем самоссылки
            ingoing_list.append(sorted(sources))
        df['Ingoing References'] = ingoing_list
    
    return dataframes_dict

print("✅ Все функции очистки загружены")

# === Применение очистки к загруженным данным ===
# Ожидаемые имена файлов (можно адаптировать под ваши)
# Для каждого нужного файла вызываем rename_first_column с соответствующим key_name
dataframes['Multicubes.csv'] = clean_df_multicubes(rename_first_column(dataframes['Multicubes.csv'], 'Multicubes'))
dataframes['Cubes.csv'] = clean_df_cubes(rename_first_column(dataframes['Cubes.csv'], 'Cubes'))
dataframes['Lists.csv'] = clean_df_lists(rename_first_column(dataframes['Lists.csv'], 'Lists'))
dataframes['ListsProperties.csv'] = clean_df_listsproperties(rename_first_column(dataframes['ListsProperties.csv'], 'ListsProperties'))
dataframes['CubesSubsets.csv'] = clean_df_сubesubsets(rename_first_column(dataframes['CubesSubsets.csv'], 'CubesSubsets'))
dataframes['ListsSubsets.csv'] = rename_first_column(dataframes['ListsSubsets.csv'], 'ListsSubsets')

# Обработка Referenced By
dataframes['Cubes.csv'] = clean_referenced_by(dataframes['Cubes.csv'], context_col='Multicube')
dataframes['ListsProperties.csv'] = clean_referenced_by(dataframes['ListsProperties.csv'], context_col='List')

# Добавление входящих ссылок
dataframes = add_ingoing_references(dataframes)

print(f"✅ Очистка завершена!")

## 📄 Часть 2: Генерация отчётов (с возможностью скачивания)

#### 2.1. Отчёт формирует список кубов и источников, на которые эти кубы ссылаются

Данные рассчитываются по значению поля `Referenced By` сточника

In [ ]:
# === Отчёт 1: cubes_with_incoming.csv (Multicube – Cubes) ===

def generate_ingoing_report_csv(df_multicubes, df_cubes):
    """
    Генерирует CSV с колонками:
    - Multicube
    - Cube
    - Ingoing References
    Разделитель: точка с запятой.
    Оставляет только записи, у которых Ingoing References не пуст.
    """
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Multicube', 'Cube', 'Ingoing References'])

    for _, row in df_multicubes.iterrows():
        mc = row['Multicubes']
        mc_refs = row.get('Ingoing References', '')
        
        # Строка мультикуба — только если у него есть ссылки
        if has_items(mc_refs):
            writer.writerow([mc, '', str(mc_refs)])
        
        # Ищем кубы, принадлежащие мультикубу
        matches = df_cubes[df_cubes['Multicube'].str.contains(mc, na=False, regex=False)]
        for _, mrow in matches.iterrows():
            cube_name = mrow['Cubes']
            cube_refs = mrow.get('Ingoing References', '')
            # Строка куба — только если у него есть ссылки
            if has_items(cube_refs):
                writer.writerow([mc, cube_name, str(cube_refs)])

    return output.getvalue()

report1_csv = generate_ingoing_report_csv(
    dataframes['Multicubes.csv'],
    dataframes['Cubes.csv']
)
download_csv(report1_csv, "cubes_with_incoming.csv")

#### 2.2. Отчёт содержит список кубов, где количество значений выше медианного
 
В отчёт отбираются кубы, которые содержат больше ячеек, чем минимум половина остальных

In [ ]:
# === Отчёт 2: big_cubes.csv (Cubes) ===
def generate_big_cubes_csv(df_cubes):
    """
    Генерирует CSV-строку с кубами, у которых Cell Count выше медианного значения.
    
    Колонки в выходном CSV:
        - Multicube
        - Cube         (берётся из колонки 'Cubes')
        - Full Name
        - Cell Count
    
    Сортировка: по убыванию Cell Count.
    Разделитель: точка с запятой (;).
    """
    # Копируем и преобразуем Cell Count в число
    df = df_cubes.copy()
    df['Cell Count'] = pd.to_numeric(df['Cell Count'], errors='coerce')
    
    # Удаляем строки, где Cell Count не число
    df_clean = df.dropna(subset=['Cell Count'])
    
    if df_clean.empty:
        # Если нет числовых значений, возвращаем только заголовок
        return "Multicube;Cube;Full Name;Cell Count\n"
    
    # Вычисляем медиану
    median_val = df_clean['Cell Count'].median()
    
    # Фильтруем значения СТРОГО выше медианы
    df_filtered = df_clean[df_clean['Cell Count'] > median_val]
    
    # Сортируем по убыванию Cell Count
    df_sorted = df_filtered.sort_values('Cell Count', ascending=False)
    
    # Выбираем нужные колонки и переименовываем 'Cubes' -> 'Cube'
    df_result = df_sorted[['Multicube', 'Cubes', 'Full Name', 'Cell Count']].copy()
    df_result.rename(columns={'Cubes': 'Cube'}, inplace=True)
    
    # Записываем в CSV (разделитель ;)
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Multicube', 'Cube', 'Full Name', 'Cell Count'])
    
    for _, row in df_result.iterrows():
        writer.writerow([
            row['Multicube'],
            row['Cube'],
            row['Full Name'],
            row['Cell Count']
        ])
    
    return output.getvalue()

report2_csv = generate_big_cubes_csv(dataframes['Cubes.csv'])
download_csv(report2_csv, "big_cubes.csv")

#### 2.3 Отчёт для классификации кубов

Кубы разделяются на 4 типа:
* Single - куб который не используется в других сущностях и сам не использует сущности
* Source - куб, который является только источником (используется в других сущностях)
* Consumer - куб, который только использует другие сущности
* Transit - куб, который одновременно является и источником, и потребителем данных из других сущностей

In [ ]:
# === Отчёт 3: cubes_classification.csv (классификация кубов) ===
def generate_cubes_classification_csv(df_cubes):
    """
    Генерирует CSV с полной классификацией всех кубов.
    Колонки:
        - Multicube
        - Cube           (из поля 'Cubes')
        - Outgoing References
        - Ingoing References
        - Type           (Single / Source / Consumer / Transit)
    Разделитель: точка с запятой.
    """
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow([
        'Multicube', 'Cube', 'Outgoing References',
        'Ingoing References', 'Type'
    ])

    for _, row in df_cubes.iterrows():
        multicube = row.get('Multicube', '')
        cube = row.get('Cubes', '')
        out_refs = row.get('Outgoing References', [])
        in_refs = row.get('Ingoing References', [])

        out_empty = not has_items(out_refs)
        in_empty = not has_items(in_refs)

        # Классификация типа
        if out_empty and in_empty:
            type_val = 'Single'
        elif not out_empty and in_empty:
            type_val = 'Source'
        elif out_empty and not in_empty:
            type_val = 'Consumer'
        else:
            type_val = 'Transit'

        # Преобразование в строку для записи (всегда включаем)
        out_str = str(out_refs)
        in_str = str(in_refs)

        writer.writerow([multicube, cube, out_str, in_str, type_val])

    return output.getvalue()

# Генерируем CSV с классификацией всех кубов
classification_csv = generate_cubes_classification_csv(dataframes['Cubes.csv'])

# Скачиваем с поддержкой кириллицы
download_csv(classification_csv, "cubes_classification.csv")

#### 2.4 Отчёт для классификации справочников

Справочники разделяются на 3 типа
* Empty - пустые, с нулевым количеством ячеек
* Used - непустые, используемые в мультикубах в качестве измерений
* Unused - непустые, не используемые в мультикубах

In [ ]:
# === Отчёт 4: lists_classification.csv (классификация справочников) ===
def generate_lists_classification_csv(df_lists, df_listsproperties):
    """
    Генерирует CSV-строку с классификацией списков.
    Колонки:
        - List
        - Cell Count
        - Type        (Empty / Unused / Used)
        - Used In     (список двучастных имён, где этот список используется как внешняя ссылка)
    Разделитель: точка с запятой.
    """
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['List', 'Cell Count', 'Type', 'Used In'])

    # Приводим Cell Count к числовому типу
    df_lists = df_lists.copy()
    df_lists['Cell Count'] = pd.to_numeric(df_lists['Cell Count'], errors='coerce')

    # Группируем свойства по полю 'List' для быстрого поиска
    props_by_list = {}
    for _, row in df_listsproperties.iterrows():
        lst = row.get('List')
        if pd.isna(lst):
            continue
        if lst not in props_by_list:
            props_by_list[lst] = []
        props_by_list[lst].append(row)

    # Обрабатываем каждый список
    for _, row in df_lists.iterrows():
        list_name = row['Lists']
        cell_count = row['Cell Count']

        # Случай Empty (Cell Count = 0 или NaN)
        if pd.isna(cell_count) or cell_count == 0:
            writer.writerow([list_name, 0, 'Empty', ''])
            continue

        # Собираем used_in – множество оригинальных ссылок, где первая часть не равна list_name
        used_in = set()
        for prop_row in props_by_list.get(list_name, []):
            out_refs = prop_row.get('Outgoing References', [])
            if not has_items(out_refs):
                continue
            # Приводим к списку, если массив NumPy
            if isinstance(out_refs, np.ndarray):
                out_refs = out_refs.tolist()
            elif not isinstance(out_refs, list):
                continue  # неожиданный тип

            for ref in out_refs:
                if not isinstance(ref, str):
                    ref = str(ref)
                if '.' not in ref:
                    continue  # одночастные ссылки не учитываем (в условии задачи только двучастные)
                first_part, _ = parse_name(ref)
                # Убираем кавычки для сравнения с именем списка
                first_clean = first_part.strip("'")
                if first_clean != list_name:
                    used_in.add(ref)

        type_val = 'Unused' if len(used_in) == 0 else 'Used'
        used_in_str = str(list(used_in)) if used_in else ''
        writer.writerow([list_name, cell_count, type_val, used_in_str])

    return output.getvalue()

# Генерируем CSV-отчёт по спискам
lists_classification = generate_lists_classification_csv(
    dataframes['Lists.csv'],
    dataframes['ListsProperties.csv']
)

# Скачиваем с поддержкой кириллицы
download_csv(lists_classification, "lists_classification.csv")

#### 2.5 Отчёт для оценки размеров кубов и измерений мультикуба

Отчёт отображает количество ячеек в мультикубах, кубах и измерениях

In [ ]:
# === Отчёт 5: advanced_multicube_report.csv (для анализа измерений) ===

def generate_advanced_multicube_csv(cubes_df, multicubes_df, lists_df, subsets_df):
    """
    Генерирует CSV-отчёт по мультикубам, содержащим кубы с Cell Count > медианного значения.
    Колонки:
        - Multicube
        - Source            (пусто для строки мультикуба, для источника – его имя)
        - Cell Count        (сумма Cell Count отфильтрованных кубов для мультикуба;
                             для источника – его Cell Count)
        - Cubes Count       (количество отфильтрованных кубов в мультикубе; для источника – пусто)
        - Cubes Cell Count  (Cell Count первого отфильтрованного куба мультикуба; для источника – пусто)
    Разделитель: точка с запятой.
    """
    # Подготовка словарей списков и сабсетов
    lists_map = {}
    if 'Lists' in lists_df.columns and 'Cell Count' in lists_df.columns:
        lists_map = lists_df.drop_duplicates('Lists').set_index('Lists')['Cell Count'].to_dict()

    subsets_map = {}
    if subsets_df is not None and 'ListsSubsets' in subsets_df.columns and 'Full Name' in subsets_df.columns:
        for _, r in subsets_df.iterrows():
            subsets_map[r['ListsSubsets']] = (r['Full Name'], r.get('Cell Count', 0))

    # Копируем и чистим кубы
    cubes_df = cubes_df.copy()
    cubes_df['Cell Count'] = pd.to_numeric(cubes_df['Cell Count'], errors='coerce')
    cubes_clean = cubes_df.dropna(subset=['Cell Count'])

    if cubes_clean.empty:
        # Нет числовых данных – возвращаем пустой CSV с заголовком
        return "Multicube;Source;Cell Count;Cubes Count;Cubes Cell Count\n"

    # Вычисляем медиану Cell Count
    median_val = cubes_clean['Cell Count'].median()

    # Фильтруем кубы, строго большие медианы
    big_cubes = cubes_clean[cubes_clean['Cell Count'] > median_val]

    if big_cubes.empty:
        return "Multicube;Source;Cell Count;Cubes Count;Cubes Cell Count\n"

    # Группировка по мультикубу
    grouped = big_cubes.groupby('Multicube').agg({
        'Cell Count': ['sum', 'count', 'first']
    }).round()
    # Приводим к int64 для WebAssembly (без переполнения)
    grouped[('Cell Count', 'sum')] = grouped[('Cell Count', 'sum')].astype('int64')
    grouped[('Cell Count', 'first')] = grouped[('Cell Count', 'first')].astype('int64')
    grouped.columns = ['Sum', 'Count', 'First']
    grouped = grouped.reset_index()
    grouped = grouped.sort_values('Sum', ascending=False)

    # Отбираем только мультикубы, попавшие в группировку
    mc_names = set(grouped['Multicube'])
    multicubes_filtered = multicubes_df[multicubes_df['Multicubes'].isin(mc_names)]

    # Функция для разбора строки с разделителями-запятыми
    def parse_comma(text):
        if not isinstance(text, str):
            return []
        return [x.strip() for x in text.split(',') if x.strip()]

    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Multicube', 'Source', 'Cell Count', 'Cubes Count', 'Cubes Cell Count'])

    for _, row in grouped.iterrows():
        mc = row['Multicube']
        mc_sum = row['Sum']
        mc_count = row['Count']
        mc_first = row['First']

        # Строка мультикуба
        writer.writerow([mc, '', mc_sum, mc_count, mc_first])

        # Находим запись мультикуба в датафрейме multicubes
        mc_row = multicubes_filtered[multicubes_filtered['Multicubes'] == mc]
        if mc_row.empty:
            continue

        # Собираем уникальные источники из полей
        items = set()
        for field in ['User Lists', 'Time Scale', 'Versions', 'Cube Subset']:
            if field in mc_row.columns:
                items.update(parse_comma(mc_row.iloc[0][field]))

        # Для каждого источника – своя строка
        for src in sorted(items):
            clean_src = src.strip("'")
            cell_val = None
            if clean_src in lists_map:
                cell_val = lists_map[clean_src]
            elif clean_src in subsets_map:
                cell_val = subsets_map[clean_src][1]
            writer.writerow([mc, src, cell_val if cell_val is not None else '', '', ''])

    return output.getvalue()

report5_csv = generate_advanced_multicube_csv(
    dataframes['Cubes.csv'],
    dataframes['Multicubes.csv'],
    dataframes['Lists.csv'],
    dataframes.get('ListsSubsets.csv')  # если есть, иначе None
)

download_csv(report5_csv, "advanced_multicube_report.csv")


#### 2.6 Отчёт о дубликатах формул

Отчет содержит список формул и список сущностей, где формула используется

In [ ]:
# === Отчёт 6: formula_duplicates.txt (формульные дубликаты)  ===
def formula_duplication_report(df_list):
    all_rows = []
    for df in df_list:
        if 'Formula' in df.columns and 'Full Name' in df.columns:
            tmp = df[['Formula', 'Full Name']].copy()
            tmp['Formula'] = tmp['Formula'].astype(str).str.strip()
            tmp = tmp[tmp['Formula'] != '']
            all_rows.append(tmp)
    if not all_rows:
        return "Нет данных для анализа формул."
    comb = pd.concat(all_rows, ignore_index=True)
    groups = comb.groupby('Formula')['Full Name'].apply(list).to_dict()
    items = [(formula, sorted(names), len(names)) for formula, names in groups.items()]
    items.sort(key=lambda x: (-x[2], x[0]))
    lines = ["📊 ОТЧЁТ: Уникальные формулы во всех кубах", "="*80, ""]
    for formula, names, cnt in items:
        lines.append(f"Formula: {formula}  (используется в {cnt} кубах)")
        for name in names:
            lines.append(f"    Full Name: {name}")
        lines.append("")
    lines.append(f"📊 Статистика: уникальных формул = {len(items)}, всего кубов с формулами = {len(comb)}")
    lines.append("=" * 80 + "\n")
    return "\n".join(lines)

report5_content = formula_duplication_report([dataframes['Cubes.csv'], dataframes['ListsProperties.csv']])
download_text(report5_content, "formula_duplicates.txt")

#### 2.7 Отчёт о суммировании в кубах

Отчет содержит список кубов, где суммирование потенциально не нужно

In [ ]:
import json


def generate_summary_format_report(cubes_df):
    """
    Генерирует CSV-отчёт по кубам, где:
    - 'Summary' == "Sum"
    - 'Format' содержит валидный JSON с:
        "dataType": "NUMBER",
        "currencyCode": null,
        "customUnits": null
    - 'Cubes' НЕ содержит ключевых слов: 'сумма', 'итого', 'расчет' (и аналоги)
    - В отчёт входят только: 'Full Name', 'Cell Count'
    - Сортировка по 'Cell Count' по убыванию

    Разделитель: точка с запятой.
    """
    # Копируем и очищаем данные
    cubes_df = cubes_df.copy()
    
    # Приводим 'Cell Count' к числу, игнорируем некорректные
    cubes_df['Cell Count'] = pd.to_numeric(cubes_df['Cell Count'], errors='coerce')
    cubes_df = cubes_df.dropna(subset=['Cell Count'])
    
    # Фильтруем строки, где 'Summary' == "Sum"
    summary_sum_df = cubes_df[cubes_df['Summary'] == "Sum"].copy()
    
    if summary_sum_df.empty:
        return "Full Name;Cell Count\n"
    
    # Список запрещённых слов (в нижнем регистре для чувствительности к регистру)
    forbidden_words = {'сумма', 'итого', 'расчет', 'итог', 'сумм', 'итоги', 'подсчёт', 'подсчет', 'всего', 'общее'}
    
    # Функция для проверки, содержит ли строка 'Cubes' запрещённые слова
    def contains_forbidden(text):
        if not isinstance(text, str):
            return False
        text_lower = text.lower()
        return any(word in text_lower for word in forbidden_words)
    
    # Фильтруем строки, где 'Cubes' НЕ содержит запрещённых слов
    summary_sum_df = summary_sum_df[~summary_sum_df['Cubes'].apply(contains_forbidden)]
    
    if summary_sum_df.empty:
        return "Full Name;Cell Count\n"
    
    # Функция для парсинга и валидации JSON в поле 'Format'
    def validate_format_json(format_str):
        try:
            fmt = json.loads(format_str)
        except (json.JSONDecodeError, TypeError):
            return False
        
        # Проверяем обязательные поля
        if not isinstance(fmt, dict):
            return False
        
        if fmt.get("dataType") != "NUMBER":
            return False
        
        if fmt.get("currencyCode") is not None:
            return False
        
        if fmt.get("customUnits") is not None:
            return False
        
        # Дополнительно: можно проверить другие поля на соответствие типам, но по ТЗ достаточно вышеуказанных
        return True
    
    # Применяем валидацию формата
    summary_sum_df = summary_sum_df[summary_sum_df['Format'].apply(validate_format_json)]
    
    if summary_sum_df.empty:
        return "Full Name;Cell Count\n"
    
    # Оставляем только нужные столбцы и сортируем по Cell Count по убыванию
    result_df = summary_sum_df[['Full Name', 'Cell Count']].sort_values('Cell Count', ascending=False)
    
    # Генерируем CSV
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Full Name', 'Cell Count'])
    
    for _, row in result_df.iterrows():
        writer.writerow([row['Full Name'], int(row['Cell Count'])])  # Приводим к int для целочисленного вывода
    
    return output.getvalue()

# Пример использования:
report7_csv = generate_summary_format_report(dataframes['Cubes.csv'])
download_csv(report7_csv, "summary_format_report.csv")

#### 2.8 Отчёт о текстовых кубах без ограничения формата

Отчет содержит список кубов, где ограничение количества знаков избыточно

In [ ]:
def generate_text_format_report(cubes_df):
    """
    Генерирует CSV-отчёт по кубам, где:
    - 'Format' содержит валидный JSON с:
        "dataType": "TEXT",
        "maxLength": 0
    - 'Referenced By' не пустой, не NaN, не null
    - В отчёт входят только: 'Full Name', 'Cell Count'
    - Сортировка по 'Cell Count' по убыванию

    Разделитель: точка с запятой.
    """
    # Копируем и очищаем данные
    cubes_df = cubes_df.copy()
    
    # Приводим 'Cell Count' к числу, игнорируем некорректные
    cubes_df['Cell Count'] = pd.to_numeric(cubes_df['Cell Count'], errors='coerce')
    cubes_df = cubes_df.dropna(subset=['Cell Count'])
    
    # Функция для валидации JSON в поле 'Format'
    def validate_text_format(format_str):
        try:
            fmt = json.loads(format_str)
        except (json.JSONDecodeError, TypeError, AttributeError):
            return False
        
        # Проверяем, что это словарь
        if not isinstance(fmt, dict):
            return False
        
        # Проверяем обязательные поля
        if fmt.get("dataType") != "TEXT":
            return False
        
        if fmt.get("maxLength") != 0:
            return False
        
        # Дополнительно: можно проверить, что "textType" и "lineBreak" присутствуют, но по ТЗ необязательно
        # Если нужно строго — добавьте: fmt.get("textType") == "GENERAL" и fmt.get("lineBreak") is True
        # Но в ТЗ указано только dataType и maxLength — поэтому оставляем только их
        
        return True
    
    # Фильтруем по валидному формату
    text_format_df = cubes_df[cubes_df['Format'].apply(validate_text_format)].copy()
    
    if text_format_df.empty:
        return "Full Name;Cell Count\n"
    
    # Фильтруем по 'Referenced By': не должен быть пустым, NaN, None, или пустой строкой
    # Убираем строки, где 'Referenced By' пусто или NaN
    text_format_df = text_format_df[
        text_format_df['Referenced By'].notna() & 
        (text_format_df['Referenced By'].str.strip() != '')
    ]
    
    if text_format_df.empty:
        return "Full Name;Cell Count\n"
    
    # Оставляем только нужные столбцы и сортируем по Cell Count по убыванию
    result_df = text_format_df[['Full Name', 'Cell Count']].sort_values('Cell Count', ascending=False)
    
    # Генерируем CSV
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Full Name', 'Cell Count'])
    
    for _, row in result_df.iterrows():
        writer.writerow([row['Full Name'], int(row['Cell Count'])])  # Целочисленный вывод
    
    return output.getvalue()

# Пример использования:
report7_csv = generate_text_format_report(dataframes['Cubes.csv'])
download_csv(report7_csv, "text_format_report.csv")

## 📈 Часть 3: Поиск сущностей

#### 3.1 Поиск Id по имени сущности

Введите имя сущности. Если имя будет найдено в данных модели, то вернется полное значение Id

In [ ]:
# === Поиск Id ===

# ---------- 1. Функция разбора имени ----------
def parse_name(s: str):
    s = str(s)
    parts = []
    i = 0
    n = len(s)
    while i < n and len(parts) < 2:
        if s[i] == "'":
            j = i + 1
            while j < n:
                if s[j] == "'" and (j + 1 == n or s[j + 1] == '.'):
                    parts.append(s[i:j+1])
                    i = j + 1
                    if i < n and s[i] == '.':
                        i += 1
                    break
                j += 1
            else:
                parts.append(s[i:])
                break
        else:
            j = i
            while j < n and s[j] != '.':
                j += 1
            parts.append(s[i:j])
            i = j
            if i < n and s[i] == '.':
                i += 1
    return parts

# ---------- 2. Утилиты для работы с Id ----------
def strip_quotes(s: str) -> str:
    if isinstance(s, str) and len(s) >= 2 and s[0] == "'" and s[-1] == "'":
        return s[1:-1]
    return s

def format_id(id_val) -> str:
    """Преобразует Id в строку без экспоненциальной записи и без .0"""
    if pd.isna(id_val):
        return ''
    try:
        num = float(str(id_val).strip())
        if abs(num - round(num)) < 1e-10:
            return str(int(round(num)))
        else:
            return f"{num:.10f}".rstrip('0').rstrip('.')
    except (ValueError, TypeError):
        return str(id_val)

# ---------- 3. Основная логика поиска (возвращает структурированный результат) ----------
def resolve_entity(name: str, dataframes: dict):
    """
    Возвращает словарь:
    {
        'type': 'single' | 'multicube_cube' | 'list_property' | 'not_found',
        'original_name': str,
        'parts': list[str],           # очищенные части
        'ids': list[str]              # соответствующие Id (1 или 2 элемента)
    }
    """
    parts = parse_name(name)
    if len(parts) == 1:
        clean_entity = strip_quotes(parts[0])
        # Мультикубы
        df = dataframes.get('Multicubes.csv')
        if df is not None and 'Multicubes' in df.columns:
            mask = df['Multicubes'] == clean_entity
            if mask.any():
                return {
                    'type': 'single',
                    'original_name': name,
                    'parts': [clean_entity],
                    'ids': [format_id(df.loc[mask, 'Id'].iloc[0])]
                }
        # Справочники
        df = dataframes.get('Lists.csv')
        if df is not None and 'Lists' in df.columns:
            mask = df['Lists'] == clean_entity
            if mask.any():
                return {
                    'type': 'single',
                    'original_name': name,
                    'parts': [clean_entity],
                    'ids': [format_id(df.loc[mask, 'Id'].iloc[0])]
                }
        # Кубы
        df = dataframes.get('Cubes.csv')
        if df is not None and 'Cubes' in df.columns:
            mask = df['Cubes'] == clean_entity
            if mask.any():
                return {
                    'type': 'single',
                    'original_name': name,
                    'parts': [clean_entity],
                    'ids': [format_id(df.loc[mask, 'Id'].iloc[0])]
                }
        # Свойства справочников
        df = dataframes.get('ListsProperties.csv')
        if df is not None and 'ListsProperties' in df.columns:
            mask = df['ListsProperties'] == clean_entity
            if mask.any():
                return {
                    'type': 'single',
                    'original_name': name,
                    'parts': [clean_entity],
                    'ids': [format_id(df.loc[mask, 'Id'].iloc[0])]
                }
        return {'type': 'not_found', 'original_name': name, 'parts': [], 'ids': []}

    elif len(parts) == 2:
        first = strip_quotes(parts[0])
        second = strip_quotes(parts[1])

        # Вариант: мультикуб.куб
        df_mc = dataframes.get('Multicubes.csv')
        df_c = dataframes.get('Cubes.csv')
        if df_mc is not None and df_c is not None:
            mask_mc = df_mc['Multicubes'] == first
            if mask_mc.any():
                id1 = format_id(df_mc.loc[mask_mc, 'Id'].iloc[0])
                mask_c = df_c['Cubes'] == second
                if mask_c.any():
                    id2 = format_id(df_c.loc[mask_c, 'Id'].iloc[0])
                    return {
                        'type': 'multicube_cube',
                        'original_name': name,
                        'parts': [first, second],
                        'ids': [id1, id2]
                    }

        # Вариант: справочник.свойство
        df_l = dataframes.get('Lists.csv')
        df_lp = dataframes.get('ListsProperties.csv')
        if df_l is not None and df_lp is not None:
            mask_l = df_l['Lists'] == first
            if mask_l.any():
                id1 = format_id(df_l.loc[mask_l, 'Id'].iloc[0])
                mask_lp = df_lp['ListsProperties'] == second
                if mask_lp.any():
                    id2 = format_id(df_lp.loc[mask_lp, 'Id'].iloc[0])
                    return {
                        'type': 'list_property',
                        'original_name': name,
                        'parts': [first, second],
                        'ids': [id1, id2]
                    }
        return {'type': 'not_found', 'original_name': name, 'parts': [], 'ids': []}
    else:
        return {'type': 'not_found', 'original_name': name, 'parts': [], 'ids': []}

# ---------- 4. Создание виджетов (один раз) ----------
if 'name_to_id_text' not in globals():
    name_to_id_text = widgets.Text(
        value='',
        placeholder='Введите имя сущности (например, my_cube или "my cube")',
        description='Имя:',
        disabled=False
    )
    name_to_id_button = widgets.Button(description='Найти Id')
    name_to_id_output = widgets.Output()

    def on_name_to_id_click(b):
        with name_to_id_output:
            name_to_id_output.clear_output()
            raw_name = name_to_id_text.value.strip()
            if not raw_name:
                print("⚠️ Введите имя.")
                return
            print(f"🔍 Введено: '{raw_name}'")
            try:
                if 'dataframes' not in globals():
                    print("❌ Переменная 'dataframes' не найдена.")
                    return
                result = resolve_entity(raw_name, dataframes)
                if result['type'] == 'not_found':
                    print(f"❌ Сущность не найдена: '{raw_name}'")
                    return

                # Формируем основной вывод (Id)
                if result['type'] == 'single':
                    main_id = result['ids'][0]
                    print(f"✅ Id: '{main_id}'")
                    # Дополнительная строка с квадратными и фигурными скобками
                    print(f"|| [{result['parts'][0]}] LONG_ID {{{main_id}}} ||")
                elif result['type'] == 'multicube_cube':
                    id1, id2 = result['ids']
                    print(f"✅ Id: '{id1}.{id2}'")
                    print(f"|| [{result['original_name']}] CUBE_VALUE {{{id1}}} {{{id2}}} ||")
                elif result['type'] == 'list_property':
                    id1, id2 = result['ids']
                    print(f"✅ Id: '{id1}.{id2}'")
                    print(f"|| [{result['parts'][0]}] LONG_ID {{{id1}}} ||")
                    print(f"|| [{result['parts'][1]}] LONG_ID {{{id2}}} ||")
            except Exception as e:
                print(f"❌ Ошибка: {e}")

    name_to_id_button.on_click(on_name_to_id_click)

# Отображение (можно выполнять несколько раз)
display(name_to_id_text, name_to_id_button, name_to_id_output)

#### 3.2 Поиск имени сущности по Id

Введите Id сущности. Если Id будет найдена в данных модели, то вернется полное значение имени

In [ ]:
# === Поиск Имени ===

# ---------- 1. Функция разбора имени (повторно, на случай если не определена) ----------
def parse_name(s: str):
    s = str(s)
    parts = []
    i = 0
    n = len(s)
    while i < n and len(parts) < 2:
        if s[i] == "'":
            j = i + 1
            while j < n:
                if s[j] == "'" and (j + 1 == n or s[j + 1] == '.'):
                    parts.append(s[i:j+1])
                    i = j + 1
                    if i < n and s[i] == '.':
                        i += 1
                    break
                j += 1
            else:
                parts.append(s[i:])
                break
        else:
            j = i
            while j < n and s[j] != '.':
                j += 1
            parts.append(s[i:j])
            i = j
            if i < n and s[i] == '.':
                i += 1
    return parts

# ---------- 2. Форматирование имени части (с учётом кириллицы) ----------
def format_name_part(part: str) -> str:
    if not isinstance(part, str):
        part = str(part)
    # Разрешены: буквы (любого алфавита), цифры, подчёркивание
    if all(c.isalpha() or c.isdigit() or c == '_' for c in part):
        return part
    else:
        return f"'{part}'"

# ---------- 3. Нормализация Id для сравнения ----------
def normalize_id(id_val) -> str:
    """Приводит Id из любого формата (float, int, строка с e) к строке-целому"""
    if pd.isna(id_val):
        return ''
    try:
        num = float(str(id_val).strip())
        if abs(num - round(num)) < 1e-10:
            return str(int(round(num)))
        else:
            return f"{num:.10f}".rstrip('0').rstrip('.')
    except (ValueError, TypeError):
        return str(id_val)

# ---------- 4. Основная логика обратного поиска ----------
def resolve_by_id(id_str: str, dataframes: dict) -> str:
    if '.' in id_str:
        first_id, second_id = id_str.split('.', 1)
        # Сценарий мультикуб.куб
        df_mc = dataframes.get('Multicubes.csv')
        df_c = dataframes.get('Cubes.csv')
        if df_mc is not None and df_c is not None:
            mask_mc = df_mc['Id'].apply(normalize_id) == first_id
            if mask_mc.any():
                first_name = df_mc.loc[mask_mc, 'Multicubes'].iloc[0]
                mask_c = df_c['Id'].apply(normalize_id) == second_id
                if mask_c.any():
                    second_name = df_c.loc[mask_c, 'Cubes'].iloc[0]
                    return f"{format_name_part(first_name)}.{format_name_part(second_name)}"
        # Сценарий справочник.свойство
        df_l = dataframes.get('Lists.csv')
        df_lp = dataframes.get('ListsProperties.csv')
        if df_l is not None and df_lp is not None:
            mask_l = df_l['Id'].apply(normalize_id) == first_id
            if mask_l.any():
                first_name = df_l.loc[mask_l, 'Lists'].iloc[0]
                mask_lp = df_lp['Id'].apply(normalize_id) == second_id
                if mask_lp.any():
                    second_name = df_lp.loc[mask_lp, 'ListsProperties'].iloc[0]
                    return f"{format_name_part(first_name)}.{format_name_part(second_name)}"
        return id_str
    else:
        # Одиночный Id
        for key, name_col in [
            ('Multicubes.csv', 'Multicubes'),
            ('Lists.csv', 'Lists'),
            ('Cubes.csv', 'Cubes'),
            ('ListsProperties.csv', 'ListsProperties')
        ]:
            df = dataframes.get(key)
            if df is not None:
                mask = df['Id'].apply(normalize_id) == id_str
                if mask.any():
                    return format_name_part(df.loc[mask, name_col].iloc[0])
        return id_str

# ---------- 5. Создание виджетов (один раз) ----------
if 'id_to_name_text' not in globals():
    id_to_name_text = widgets.Text(
        value='',
        placeholder='Введите Id (например, 102000000139 или 102000000139.392000001504)',
        description='Id:',
        disabled=False
    )
    id_to_name_button = widgets.Button(description='Найти имя')
    id_to_name_output = widgets.Output()

    def on_id_to_name_click(b):
        with id_to_name_output:
            id_to_name_output.clear_output()
            raw_id = id_to_name_text.value.strip()
            if not raw_id:
                print("⚠️ Введите Id.")
                return
            print(f"🔍 Введён Id: '{raw_id}'")
            try:
                if 'dataframes' not in globals():
                    print("❌ Переменная 'dataframes' не найдена.")
                    return
                result = resolve_by_id(raw_id, dataframes)
                print(f"✅ Имя: '{result}'")
            except Exception as e:
                print(f"❌ Ошибка: {e}")

    id_to_name_button.on_click(on_id_to_name_click)

display(id_to_name_text, id_to_name_button, id_to_name_output)

## 📈 Часть 4: Анализ формул и формульных зависимостей

In [ ]:
# === 1. Подготовка глобального маппинга по Id ===
id_lookup_map = {}  # key: Id (str) → value: {'Full Name': str, 'Source': str}
all_id_sources = {}  # key: df_name → df (с cleaned Id и Full Name)

print("🔍 Ищем датафреймы с полями 'Id' и 'Full Name'...")

for df_name, df in dataframes.items():
    if 'Id' in df.columns and 'Full Name' in df.columns:
        # Копируем и очищаем
        df_clean = df[['Id', 'Full Name']].copy()
        df_clean['Id'] = df_clean['Id'].astype(str).str.strip()
        df_clean['Full Name'] = df_clean['Full Name'].astype(str).str.strip()
        df_clean = df_clean.dropna(subset=['Id', 'Full Name'])

        # Убираем дубликаты Id (оставляем первое вхождение)
        df_clean = df_clean.drop_duplicates(subset=['Id'], keep='first')

        # Добавляем в глобальный lookup
        for _, row in df_clean.iterrows():
            id_val = row['Id']
            if id_val and id_val not in id_lookup_map:
                id_lookup_map[id_val] = {
                    'Full Name': row['Full Name'],
                    'Source': df_name
                }

        all_id_sources[df_name] = df_clean
        print(f"  ✅ {df_name}: {len(df_clean)} уникальных записей (Id + Full Name)")

print(f"✅ Всего уникальных Id в системе: {len(id_lookup_map)}")

# === 2. Создание парсера формул ===
def make_new_parser():
    """
    Парсит формулы, ищет 12-значные Id в одинарных кавычках.
    Возвращает маппинг найденных сущностей по Id.
    """
    # Регулярное выражение: одинарные кавычки + ровно 12 цифр
    _id_pattern = re.compile(r"'(\d{12})'")

    def parse_formula(stable_view_formula, full_name, source_df_name):
        """
        Парсит **Stable View Formula** и возвращает структурированные данные.
        Formula — не используется для парсинга, только для фильтрации.
        """
        if not isinstance(stable_view_formula, str) or pd.isna(stable_view_formula):
            return {
                'ParsedReferences': [],
                'ParsedIds': [],
                'ParsedSources': []
            }

        # Извлекаем все совпадения (12-значные Id)
        found_ids = _id_pattern.findall(stable_view_formula)
        # Убираем дубликаты, сохраняя порядок первого вхождения
        seen = set()
        unique_ids = []
        for id_val in found_ids:
            if id_val not in seen:
                seen.add(id_val)
                unique_ids.append(id_val)

        references = []      # Full Name найденных сущностей
        sources_data = []    # Список объектов: {'Full Name', 'Id', 'Source'}

        for id_val in unique_ids:
            if id_val in id_lookup_map:
                entity = id_lookup_map[id_val]
                references.append(entity['Full Name'])
                sources_data.append({
                    'Full Name': entity['Full Name'],
                    'Id': id_val,
                    'Source': entity['Source']
                })

        # Убираем дубликаты Full Name в ParsedReferences (сохраняя порядок)
        unique_references = []
        seen_refs = set()
        for ref in references:
            if ref not in seen_refs:
                seen_refs.add(ref)
                unique_references.append(ref)

        return {
            'ParsedReferences': unique_references,
            'ParsedIds': unique_ids,
            'ParsedSources': sources_data
        }

    return parse_formula

# === 3. Создание парсера ===
new_parser = make_new_parser()

# === 4. Обработка датафреймов с обеими формулами ===
target_dfs = []
for df_name, df in dataframes.items():
    if 'Formula' in df.columns and 'Stable View Formula' in df.columns:
        target_dfs.append((df_name, df))

print(f"✅ Найдено {len(target_dfs)} датафреймов с обеими формулами: {[name for name, _ in target_dfs]}")

df_graph = pd.DataFrame()

for df_name, df in target_dfs:
    # Создаём копию исходного датафрейма
    temp_df = df.copy()

    # ✅ ФИЛЬТР 1: Удаляем строки, где Formula NaN или пустая
    temp_df = temp_df.dropna(subset=['Formula'])
    temp_df = temp_df[temp_df['Formula'].astype(str).str.strip() != '']

    # ✅ ФИЛЬТР 2: Удаляем строки, где Stable View Formula NaN или пустая
    temp_df = temp_df.dropna(subset=['Stable View Formula'])
    temp_df = temp_df[temp_df['Stable View Formula'].astype(str).str.strip() != '']

    # Если после фильтрации ничего не осталось — пропускаем
    if len(temp_df) == 0:
        print(f"  ⚠️ {df_name}: все строки имеют пустую Formula или Stable View Formula — пропущены.")
        continue

    # ✅ Переименовываем 'Stable View Formula' в 'Formula' для единообразия в выходных данных
    # (но парсинг — только по исходному 'Stable View Formula')
    temp_df = temp_df.rename(columns={'Stable View Formula': 'Formula'})

    # Устанавливаем тип источника (имя датафрейма)
    temp_df['Type'] = df_name

    # ✅ ПАРСИНГ: применяем парсер к **исходному** 'Stable View Formula' — но он уже переименован!
    # → Мы сохранили **исходные значения** в новом столбце 'Formula', но для парсинга нужно **использовать исходные данные**
    # → Решение: сохраним исходные значения перед rename

    # ✅ ВАЖНО: Мы переименовали 'Stable View Formula' → 'Formula', но парсинг должен быть по **оригинальным значениям**
    # → Поэтому: сохраним их в отдельный столбец перед rename
    temp_df['OriginalStableViewFormula'] = df['Stable View Formula']  # Сохраняем до rename

    # Теперь парсим по оригиналу
    parsed_results = temp_df.apply(
        lambda row: new_parser(
            stable_view_formula=row['OriginalStableViewFormula'],  # ← Используем оригинальные значения
            full_name=row['Full Name'],
            source_df_name=df_name
        ),
        axis=1
    )

    # Распаковываем результаты в новые столбцы
    temp_df['ParsedReferences'] = parsed_results.apply(lambda x: x['ParsedReferences'])
    temp_df['ParsedIds'] = parsed_results.apply(lambda x: x['ParsedIds'])
    temp_df['ParsedSources'] = parsed_results.apply(lambda x: x['ParsedSources'])

    # Удаляем временный столбец (опционально)
    temp_df = temp_df.drop(columns=['OriginalStableViewFormula'])

    # ✅ КРИТИЧЕСКИЙ ШАГ 1: Сбрасываем индекс строк
    temp_df = temp_df.reset_index(drop=True)

    # ✅ КРИТИЧЕСКИЙ ШАГ 2: Убедимся, что имена столбцов уникальны и в одном регистре
    temp_df.columns = temp_df.columns.astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    temp_df = temp_df.loc[:, ~temp_df.columns.duplicated()]

    # ✅ КРИТИЧЕСКИЙ ШАГ 3: Выравниваем столбцы с df_graph
    if len(df_graph) == 0:
        df_graph = temp_df.copy()
    else:
        # Добавляем отсутствующие столбцы из df_graph в temp_df
        for col in df_graph.columns:
            if col not in temp_df.columns:
                temp_df[col] = [[] if col in ['ParsedReferences', 'ParsedIds', 'ParsedSources', 'Id'] else '' for _ in range(len(temp_df))]

        # Удаляем столбцы из temp_df, которых нет в df_graph
        temp_df = temp_df[df_graph.columns]

        # Конкатенируем
        df_graph = pd.concat([df_graph, temp_df], ignore_index=True)

# === 5. Финальная очистка: гарантированно все нужные столбцы ===
required_columns = [
    'Full Name',
    'Formula',           # ← Это переименованный 'Stable View Formula' — теперь в нем парсинговые формулы
    'Type',
    'ParsedReferences',
    'ParsedIds',
    'ParsedSources',
    'Id'
]

# Убедимся, что все нужные столбцы существуют — добавляем отсутствующие
for col in required_columns:
    if col not in df_graph.columns:
        df_graph[col] = [[] if col in ['ParsedReferences', 'ParsedIds', 'ParsedSources', 'Id'] else '' for _ in range(len(df_graph))]

# Оставляем только нужные столбцы в нужном порядке
df_graph = df_graph[required_columns]

# === 6. Вывод статистики ===
total_rows = len(df_graph)
rows_with_refs = len(df_graph[df_graph['ParsedReferences'].map(len) > 0])

print(f"\n🎉 ОБРАБОТКА ЗАВЕРШЕНА!")
print(f"   Обработано строк (с непустой Formula и Stable View Formula): {total_rows}")
print(f"   Строк с найденными зависимостями: {rows_with_refs}")
print(f"   Всего уникальных Id в системе: {len(id_lookup_map)}")
print(f"   Итоговый df_graph имеет {len(df_graph.columns)} столбцов: {list(df_graph.columns)}")

# === 7. (Опционально) Показать пример результата ===
if len(df_graph) > 0:
    print("\n📌 Пример результата (первая строка):")
    example = df_graph.iloc[0]
    print(f"  Full Name: {example['Full Name']}")
    print(f"  Formula (Stable View Formula): {example['Formula']}")
    print(f"  Type: {example['Type']}")
    print(f"  ParsedIds: {example['ParsedIds']}")
    print(f"  ParsedReferences: {example['ParsedReferences']}")
    print(f"  ParsedSources: {example['ParsedSources']}")
    print(f"  Id: {example['Id']}")

#### 4.1 Отчёт сущностей в формулах

Содержит список сущностей, извлеченных из формулы куба

In [ ]:
# === Отчёт 7: cube_references.csv (формульные дубликаты)  ===
def generate_cube_references_csv(df_graph):
    """
    Генерирует CSV-отчёт о кубах и используемых ими ссылках.
    Колонки:
        - Full Name
        - Formula
        - Parsed References
    Только кубы с непустыми ParsedReferences.
    Сортировка: по убыванию количества ссылок.
    Разделитель: точка с запятой.
    """
    # Фильтруем: оставляем только строки с непустыми ссылками
    filtered = df_graph[df_graph['ParsedReferences'].map(len) > 0].copy()
    if filtered.empty:
        return "Full Name;Formula;Parsed References\n"
    
    # Сортируем по длине списка ссылок (убывание)
    filtered = filtered.sort_values('ParsedReferences', key=lambda x: x.map(len), ascending=False)
    
    output = io.StringIO()
    writer = csv.writer(output, delimiter=';', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['Full Name', 'Formula', 'Parsed References', 'Type'])
    
    for _, row in filtered.iterrows():
        cube = row['Full Name']
        formula = row.get('Formula', '')
        refs = row['ParsedReferences']
        typs = row['Type']
        # Преобразуем список в строку (например, str(refs))
        refs_str = str(refs)
        writer.writerow([cube, formula, refs_str, typs])
    
    return output.getvalue()

report7_csv = generate_cube_references_csv(df_graph)
download_csv(report7_csv, "cube_references.csv")

#### 4.2 Отчёт о размерности известных сущностей в формулах

Отображает количество ячеек в известных сущностях формул

In [ ]:
# === Отчёт 8: entities_report.txt (сущности в формулах)  ===

from collections import OrderedDict
import math

def build_full_name_mapping(dataframes):
    """
    Собирает словарь {Full Name: Cell Count} из всех датафреймов,
    у которых есть колонки 'Full Name' и 'Cell Count'.
    Если один Full Name встречается в нескольких местах, берётся первое значение.
    """
    mapping = {}
    for name, df in dataframes.items():
        if 'Full Name' in df.columns and 'Cell Count' in df.columns:
            for _, row in df.iterrows():
                fn = row['Full Name']
                if pd.notna(fn) and fn != '' and fn not in mapping:
                    try:
                        cnt = float(row['Cell Count'])
                        if math.isnan(cnt):
                            cnt = 0
                        mapping[fn] = int(cnt)
                    except (ValueError, TypeError):
                        mapping[fn] = 0
    return mapping

def product_report(df_graph, full_name_mapping):
    lines = ["📊 ОТЧЁТ: Произведение Cell Count зависимостей по кубам", "=" * 80, ""]
    
    cubes_with_refs = df_graph[df_graph['ParsedReferences'].map(len) > 0]
    entries = []
    
    for _, row in cubes_with_refs.iterrows():
        cube_name = row['Full Name']
        refs = row['ParsedReferences']
        found = {}
        for ref in refs:
            if ref in full_name_mapping and ref not in found:
                found[ref] = full_name_mapping[ref]
        if not found:
            continue
        product = 1
        for cnt in found.values():
            product += cnt
        entries.append((product, cube_name, found))
    
    # Сортировка по убыванию произведения
    entries.sort(key=lambda x: x[0], reverse=True)
    
    for product, cube_name, found in entries:
        product_str = f"{product:,}"
        lines.append(f"{cube_name} ({product_str})")
        for ref, cnt in found.items():
            cnt_str = f"{cnt:,}"
            lines.append(f"    {ref} ({cnt_str})")
        lines.append("")
        lines.append("=" * 80 + "\n")
    
    if not entries:
        lines.append("Нет кубов с найденными зависимостями.")
    
    return "\n".join(lines)

# Использование
full_name_mapping = build_full_name_mapping(dataframes)
report_product = product_report(df_graph, full_name_mapping)
download_text(report_product, "entities_report.txt")

In [ ]:
# === Построение графа ===
full_cube_names = df_graph['Full Name']

G = nx.DiGraph()
for node in full_cube_names:
    G.add_node(node)
for _, row in df_graph.iterrows():
    src = row['Full Name']
    for target in row['ParsedReferences']:
        G.add_edge(target, src)   # источник → потребитель
print(f"✅ Граф построен: узлов = {len(G.nodes)}, рёбер = {len(G.edges)}")

In [ ]:
# === Проверка на циклы ===
double_nodes = {n for n in G.nodes() if isinstance(n, str) and n.count('.') == 1}
G_double = G.subgraph(double_nodes).copy()
if nx.is_directed_acyclic_graph(G_double):
    print("✅ Циклы между кубами не обнаружены.")
else:
    cycles = list(nx.simple_cycles(G_double))
    print(f"⚠️ Обнаружены циклы ({len(cycles)}):")
    for cyc in cycles[:5]:
        print(" → ".join(cyc + [cyc[0]]))

## ✳ Построение связей куба графически

Введите в поле "Ввод" полное имя куба (`'Имя мультикуба'.'Имя куба'`) и нажмите "Применить"

In [ ]:
# Создаём текстовое поле
text_input = widgets.Text(
    value='',           # пустая строка по умолчанию
    placeholder='Введите значение...',
    description='Ввод:',
    disabled=False
)

button = widgets.Button(description='Применить')
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()
        val = text_input.value
        print(f"Получено значение: '{val}'")
        # === Визуализация окрестности выбранного куба ===
        target_cube = val
        if target_cube in G.nodes:
            # Получаем окрестность глубиной 1
            neighbors = set(G.predecessors(target_cube)) | set(G.successors(target_cube))
            neighbors.add(target_cube)
            subG = G.subgraph(neighbors).copy()
            plt.figure(figsize=(12, 8))
            pos = nx.spring_layout(subG, seed=42)
            node_colors = ['gold' if n == target_cube else 'lightblue' for n in subG.nodes()]
            nx.draw(subG, pos, with_labels=True, node_color=node_colors, node_size=2000, font_size=8, arrows=True)
            plt.title(f"Окрестность куба {target_cube}")
            plt.show()
        else:
            print(f"Куб '{target_cube}' не найден в графе. Доступные кубы (первые 10): {list(G.nodes)[:10]}")
        

button.on_click(on_button_click)
display(text_input, button, output)
